<a href="https://colab.research.google.com/github/JagdishMane/aai-machine-learning/blob/main/vertex_ai_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Assignment 6.1 - Working with Third Party LLM APIs
### Generate samples, evaluate quality with an LLM judge, and visualize results


**Overview:**  
This notebook walks through the full pipeline:
1. Set up Google Cloud
2. Generate 50 user question + model response pairs on a chosen topic
3. Use an LLM as a judge to score each response
4. Visualize the results
5. Reflect on the effectiveness of this approach

**Model used:** `gemini-1.5-flash` (fast and cost-effective)  
**Estimated cost:** < $1 USD

---
## 📦 Step 1: Install Dependencies

In [2]:
# Run this cell first — installs all required packages
!pip install google-cloud-aiplatform vertexai pandas matplotlib seaborn tqdm --quiet

---
## 🔐 Step 2: Authenticate with Google Cloud


In [5]:
import sys
from google.colab import auth
auth.authenticate_user()    ### This authenticates colab notebook to Google Cloud

---
## ⚙️ Step 3: Configure Your Project

👉 **Replace `YOUR_PROJECT_ID` below with your actual GCP project ID.**  
Find it at: https://console.cloud.google.com → top dropdown

In [7]:
import vertexai## Autnentict

from vertexai.generative_models import GenerativeModel
import pandas as pd
import json
import time
import re

### Set GOOGLE PROJECT ID and MODEL
PROJECT_ID = "graphite-earth-293910"
LOCATION   = "us-central1"
MODEL_NAME = "gemini-1.5-flash"   # gemini-1.5-pro
TOPIC      = "USD Assignment"
NUM_SAMPLES = 50

vertexai.init(project=PROJECT_ID, location=LOCATION)   ### Initialize vertexai
model = GenerativeModel(MODEL_NAME)
print(f'✅ Vertex AI initialized | Project: {PROJECT_ID} | Model: {MODEL_NAME}')

✅ Vertex AI initialized | Project: graphite-earth-293910 | Model: gemini-1.5-flash


---
## 💬 Step 4: Generate 50 User Questions

In [8]:
def safe_json_parse(text):
    """Strip markdown fences and parse JSON safely."""
    text = re.sub(r'```(?:json)?', '', text).strip().rstrip('`').strip()
    return json.loads(text)

question_prompt = f"""Generate exactly {NUM_SAMPLES} diverse user questions about {TOPIC}.
Include a mix of:
- 15 beginner-level questions (fundamental concepts)
- 20 intermediate questions (applied techniques)
- 15 advanced questions (theory, trade-offs, edge cases)

Return ONLY a valid JSON array of {NUM_SAMPLES} strings. No explanation, no markdown.
Example format: ["What is overfitting?", "How does gradient descent work?", ...]"""

print('⏳ Generating questions...')
q_response = model.generate_content(question_prompt)
questions = safe_json_parse(q_response.text)

# Validate
assert isinstance(questions, list), "Expected a list of questions"
questions = questions[:NUM_SAMPLES]  # cap at 50 just in case

print(f'✅ Generated {len(questions)} questions.\n')
print('Sample questions:')
for i, q in enumerate(questions[:5], 1):
    print(f'  {i}. {q}')
print('  ...')

⏳ Generating questions...


PermissionDenied: 403 Agent Platform API has not been used in project graphite-earth-293910 before or it is disabled. Enable it by visiting https://console.developers.google.com/apis/api/aiplatform.googleapis.com/overview?project=graphite-earth-293910 then retry. If you enabled this API recently, wait a few minutes for the action to propagate to our systems and retry. [reason: "SERVICE_DISABLED"
domain: "googleapis.com"
metadata {
  key: "service"
  value: "aiplatform.googleapis.com"
}
metadata {
  key: "serviceTitle"
  value: "Agent Platform API"
}
metadata {
  key: "containerInfo"
  value: "graphite-earth-293910"
}
metadata {
  key: "consumer"
  value: "projects/graphite-earth-293910"
}
metadata {
  key: "activationUrl"
  value: "https://console.developers.google.com/apis/api/aiplatform.googleapis.com/overview?project=graphite-earth-293910"
}
, locale: "en-US"
message: "Agent Platform API has not been used in project graphite-earth-293910 before or it is disabled. Enable it by visiti

---
## 🤖 Step 5: Generate Model Responses

This will make 50 API calls. Expect ~2–3 minutes.

In [ ]:
from tqdm import tqdm

def generate_response(question, max_retries=3):
    """Generate a model answer with retry logic."""
    prompt = f"""Answer the following question about {TOPIC} clearly and accurately.
Be concise but complete. Aim for 3-6 sentences.

Question: {question}"""
    for attempt in range(max_retries):
        try:
            response = model.generate_content(prompt)
            return response.text.strip()
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)  # exponential backoff
            else:
                return f"ERROR: {str(e)}"

records = []
print('⏳ Generating responses (this takes ~2-3 minutes)...')

for i, question in enumerate(tqdm(questions, desc='Generating')):
    response_text = generate_response(question)
    records.append({
        'id': i + 1,
        'question': question,
        'response': response_text
    })
    time.sleep(0.5)  # gentle rate limiting

df = pd.DataFrame(records)
df.to_csv('samples.csv', index=False)
print(f'\n✅ Generated and saved {len(df)} samples to samples.csv')
df.head(3)

---
## ⚖️ Step 6: LLM-as-Judge Evaluation

Each response is rated on 4 criteria (scale of 1–5):
- **accuracy** — Is the information factually correct?
- **clarity** — Is it easy to understand?
- **completeness** — Does it fully address the question?
- **conciseness** — Is it appropriately brief without sacrificing quality?

This will make another 50 API calls (~2–3 minutes).

In [ ]:
JUDGE_CRITERIA = ['accuracy', 'clarity', 'completeness', 'conciseness']

def judge_response(question, response, max_retries=3):
    """Use the LLM to evaluate a response on 4 criteria."""
    judge_prompt = f"""You are an expert evaluator assessing the quality of an AI model's answer.

Topic domain: {TOPIC}
User question: {question}
Model's answer: {response}

Rate the answer on each criterion from 1 (very poor) to 5 (excellent):
- accuracy: Is the information factually correct and not misleading?
- clarity: Is the explanation clear and easy to understand?
- completeness: Does it fully address all parts of the question?
- conciseness: Is it appropriately concise without losing important detail?

Return ONLY a valid JSON object with these exact keys:
accuracy, clarity, completeness, conciseness, overall_score, brief_feedback

overall_score should be the average of the four criteria (rounded to 2 decimal places).
brief_feedback should be a single sentence explaining the main strength or weakness.
No markdown, no extra text."""

    for attempt in range(max_retries):
        try:
            result = model.generate_content(judge_prompt)
            parsed = safe_json_parse(result.text)
            # Validate expected keys
            for key in JUDGE_CRITERIA + ['overall_score', 'brief_feedback']:
                assert key in parsed, f'Missing key: {key}'
            return parsed
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
            else:
                return {
                    'accuracy': None, 'clarity': None,
                    'completeness': None, 'conciseness': None,
                    'overall_score': None,
                    'brief_feedback': f'Judge error: {str(e)}'
                }

judgments = []
print('⏳ Running LLM judge on all 50 responses...')

for _, row in tqdm(df.iterrows(), total=len(df), desc='Judging'):
    judgment = judge_response(row['question'], row['response'])
    judgment['id'] = row['id']
    judgments.append(judgment)
    time.sleep(0.5)

scores_df = pd.DataFrame(judgments)
final_df = df.merge(scores_df, on='id')
final_df.to_csv('evaluated_samples.csv', index=False)

print(f'\n✅ Evaluation complete. Saved to evaluated_samples.csv')
print('\nScore summary:')
print(final_df[JUDGE_CRITERIA + ['overall_score']].describe().round(2))

---
## 📊 Step 7: Visualizations

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np

sns.set_theme(style='whitegrid', palette='muted')
COLORS = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle(f'LLM-as-Judge Evaluation Results\nTopic: {TOPIC.title()}',
             fontsize=16, fontweight='bold', y=1.01)

# --- Plot 1: Distribution of Overall Scores ---
ax1 = axes[0, 0]
ax1.hist(final_df['overall_score'].dropna(), bins=12, color='#4C72B0',
         edgecolor='white', linewidth=0.8)
ax1.axvline(final_df['overall_score'].mean(), color='red',
            linestyle='--', linewidth=1.5, label=f"Mean: {final_df['overall_score'].mean():.2f}")
ax1.set_title('Distribution of Overall Scores', fontweight='bold')
ax1.set_xlabel('Overall Score (1–5)')
ax1.set_ylabel('Count')
ax1.set_xlim(1, 5)
ax1.legend()

# --- Plot 2: Average Score per Criterion ---
ax2 = axes[0, 1]
means = final_df[JUDGE_CRITERIA].mean()
bars = ax2.bar(JUDGE_CRITERIA, means, color=COLORS, edgecolor='white', linewidth=0.8)
ax2.set_ylim(0, 5.5)
ax2.set_title('Average Score per Criterion', fontweight='bold')
ax2.set_ylabel('Mean Score (1–5)')
for bar, val in zip(bars, means):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             f'{val:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# --- Plot 3: Boxplot per Criterion ---
ax3 = axes[0, 2]
bp = ax3.boxplot(
    [final_df[c].dropna() for c in JUDGE_CRITERIA],
    labels=JUDGE_CRITERIA,
    patch_artist=True,
    medianprops=dict(color='black', linewidth=2)
)
for patch, color in zip(bp['boxes'], COLORS):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax3.set_ylim(0.5, 5.5)
ax3.set_title('Score Distribution per Criterion', fontweight='bold')
ax3.set_ylabel('Score (1–5)')

# --- Plot 4: Correlation Heatmap ---
ax4 = axes[1, 0]
corr = final_df[JUDGE_CRITERIA].corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, annot=True, fmt='.2f', ax=ax4, cmap='Blues',
            linewidths=0.5, annot_kws={'size': 11}, vmin=0, vmax=1)
ax4.set_title('Criterion Correlation Heatmap', fontweight='bold')

# --- Plot 5: Score vs Question Index (trend) ---
ax5 = axes[1, 1]
ax5.plot(final_df['id'], final_df['overall_score'],
         color='#4C72B0', alpha=0.5, linewidth=0.8, marker='o', markersize=3)
# Rolling average
rolling = final_df['overall_score'].rolling(window=7, center=True).mean()
ax5.plot(final_df['id'], rolling, color='red', linewidth=2, label='7-sample rolling avg')
ax5.set_ylim(1, 5)
ax5.set_title('Overall Score Across 50 Samples', fontweight='bold')
ax5.set_xlabel('Sample ID')
ax5.set_ylabel('Overall Score (1–5)')
ax5.legend()

# --- Plot 6: Score Tier Breakdown ---
ax6 = axes[1, 2]
def tier(score):
    if score is None or pd.isna(score): return 'Unknown'
    if score >= 4.0: return 'High (4–5)'
    if score >= 3.0: return 'Medium (3–4)'
    return 'Low (1–3)'

final_df['tier'] = final_df['overall_score'].apply(tier)
tier_counts = final_df['tier'].value_counts()
tier_colors = {'High (4–5)': '#55A868', 'Medium (3–4)': '#DD8452', 'Low (1–3)': '#C44E52'}
bar_colors = [tier_colors.get(t, '#888') for t in tier_counts.index]
ax6.bar(tier_counts.index, tier_counts.values, color=bar_colors, edgecolor='white')
ax6.set_title('Response Quality Tiers', fontweight='bold')
ax6.set_ylabel('Number of Responses')
for i, (label, count) in enumerate(tier_counts.items()):
    ax6.text(i, count + 0.3, str(count), ha='center', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.savefig('evaluation_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Visualization saved to evaluation_results.png')

---
## 🔍 Step 8: Explore Notable Samples

Let's look at the best and worst rated responses to build intuition.

In [ ]:
pd.set_option('display.max_colwidth', 300)

# Top 5 highest scoring responses
print('='*70)
print('🏆 TOP 5 HIGHEST SCORING RESPONSES')
print('='*70)
top5 = final_df.nlargest(5, 'overall_score')[['id','question','response','overall_score','brief_feedback']]
for _, row in top5.iterrows():
    print(f"\n[ID {row['id']}] Score: {row['overall_score']}")
    print(f"Q: {row['question']}")
    print(f"A: {row['response'][:200]}...")
    print(f"Judge: {row['brief_feedback']}")
    print('-'*60)

print('\n')
print('='*70)
print('⚠️  BOTTOM 5 LOWEST SCORING RESPONSES')
print('='*70)
bot5 = final_df.nsmallest(5, 'overall_score')[['id','question','response','overall_score','brief_feedback']]
for _, row in bot5.iterrows():
    print(f"\n[ID {row['id']}] Score: {row['overall_score']}")
    print(f"Q: {row['question']}")
    print(f"A: {row['response'][:200]}...")
    print(f"Judge: {row['brief_feedback']}")
    print('-'*60)

---
## 📋 Step 9: Full Summary Statistics

In [ ]:
print('📊 FULL EVALUATION SUMMARY')
print('='*50)
print(f"Topic: {TOPIC}")
print(f"Model: {MODEL_NAME}")
print(f"Total samples: {len(final_df)}")
print()

summary = final_df[JUDGE_CRITERIA + ['overall_score']].agg(['mean', 'std', 'min', 'max']).round(3)
print(summary.to_string())

print()
print('Quality tier breakdown:')
print(final_df['tier'].value_counts().to_string())
print()
print(f"Responses scoring >= 4.0: {(final_df['overall_score'] >= 4.0).sum()} ({(final_df['overall_score'] >= 4.0).mean()*100:.1f}%)")
print(f"Responses scoring < 3.0:  {(final_df['overall_score'] < 3.0).sum()} ({(final_df['overall_score'] < 3.0).mean()*100:.1f}%)")

---
## ✍️ Step 10: Written Reflection

**Fill in this section with your own observations after running the notebook.**

---

### 1. Overview

> Describe the topic you chose and why. Explain the pipeline: generate questions → generate responses → LLM judge → visualize.

*[Your answer here]*

---

### 2. Observations on Generated Samples

> - Were questions diverse across difficulty levels?
> - Did the model generate longer responses for harder questions?
> - Did you notice any patterns in how the model approached certain types of questions?

*[Your answer here]*

---

### 3. LLM Judge Effectiveness

> - Do the judge's scores match your own reading of the responses?
> - Was the judge consistent across similar questions?
> - Did the judge's `brief_feedback` seem insightful or generic?
> - Which criterion showed the most variance and why?

*[Your answer here]*

---

### 4. Key Limitations Identified

> Consider the following and comment on what you observed:
> - **Self-evaluation bias**: Same model family generating and judging — did you notice any leniency?
> - **Lack of ground truth**: Without a reference answer, how confident are you in the scores?
> - **Prompt sensitivity**: How might a different judge prompt change the results?
> - **Criteria subjectivity**: E.g., is `completeness` well-defined enough for an LLM to judge reliably?

*[Your answer here]*

---

### 5. Has This Exercise Changed Your Perspective?

> Reflect on the broader question: **Is using an LLM as a judge a good idea?**  
> Under what circumstances would you trust these scores?  
> When would you NOT rely on them?  
> How does this compare to human evaluation?

*[Your answer here]*

---
## 💾 Step 11: Download Outputs

Run this cell to download your CSV and visualization if running in Google Colab.

In [ ]:
if 'google.colab' in sys.modules:
    from google.colab import files
    files.download('evaluated_samples.csv')
    files.download('evaluation_results.png')
    print('✅ Files downloaded.')
else:
    print('Files saved locally:')
    print('  - evaluated_samples.csv')
    print('  - evaluation_results.png')

---
## 🎉 Done!

**Deliverables checklist:**
- [x] GCP / Vertex AI set up
- [x] Topic chosen
- [x] 50 samples generated (`samples.csv`)
- [x] LLM judge scores computed (`evaluated_samples.csv`)
- [x] Visualizations created (`evaluation_results.png`)
- [ ] Written reflection completed (Step 10)

---
*Built with Vertex AI Gemini 1.5 Flash — estimated cost: < $1 USD*